In [1]:
# ================================================================
# ICH PATIENT-LEVEL DEEP LEARNING PIPELINE — FINAL PUBLICATION VERSION — DEVICE-SAFE
# (FIXED: efficiency/latency columns flattened to scalars — see notes below)
#
# MODELS
#   Lightweight: MobileNetV2, EfficientNet-B0
#   Larger CNNs: ResNet-50, DenseNet-121
#
# DESIGN
#   45 patients total
#   9 patients frozen held-out test (4 ICH, 5 normal)
#   36 development patients (14 ICH, 22 normal)
#   4-fold patient-level StratifiedKFold: 27 train / 9 validation
#   Primary model-selection endpoint: patient-level AUROC
#   Primary patient aggregation: mean slice probability
#   Secondary aggregation: top-10% mean and maximum
#
# TRAINING
#   ImageNet-pretrained models
#   ImageNet normalization
#   Patient-balanced weighted BCEWithLogitsLoss
#   Adam, lr=1e-4, weight_decay=1e-4
#   ReduceLROnPlateau on validation patient AUROC
#   Early stopping patience=10
#   Final epochs = median fold-best epoch
#
# OUTPUT
#   ONE MASTER CSV containing all tabular results
#   JSON protocol
#   Model checkpoints
#   Publication PNG + vector PDF figures
#   Grad-CAM figures + faithfulness curves
#   ONE ZIP containing everything
#
# IMPORTANT
#   The held-out test set is never used for architecture, epoch,
#   threshold, or model selection.
#   Grad-CAM is saliency, not lesion segmentation.
#   The 9-patient test set is very small; report uncertainty.
#   A Kaggle notebook cannot choose physical GPU hardware. This script
#   requires CUDA and uses the first available CUDA GPU; the notebook
#   runtime itself must be configured as GPU (e.g. NVIDIA T4) in Kaggle.
#
# ----------------------------------------------------------------
# WHAT WAS FIXED IN THIS VERSION (nothing else was changed)
#   `measure_latency()` returns a dict of stats (mean/median/sd/p95).
#   The original script stored that whole dict directly into a single
#   "gpu_latency_ms_per_image" / "cpu_latency_ms_per_image" cell of the
#   efficiency row. That is fine for building the row, but it breaks the
#   very next `efficiency_df.groupby("model")[...].mean()` call used for
#   the CPU-latency figure (FIG7), because pandas cannot average a column
#   of dicts:
#       TypeError: complex() first argument must be a string or a
#       number, not 'dict'
#   This is exactly the crash you hit, right after DenseNet-121 finished
#   its held-out evaluation (training, CV, held-out test, and Grad-CAM had
#   already completed successfully — the failure was purely in the later
#   efficiency plotting step). The fix flattens each latency dict into
#   four scalar columns (mean_ms, median_ms, sd_ms, p95_ms) for both GPU
#   and CPU, and FIG7 now reads the scalar "cpu_latency_mean_ms" column.
#   No splits, folds, seeds, models, losses, schedulers, thresholds,
#   aggregations, Grad-CAM logic, or figures were removed or altered.
#
#   A second, purely additive change: a safe, no-op-if-offline attempt to
#   `pip install thop` is made at startup so FLOPs/MACs can actually be
#   computed instead of silently staying NaN (your log showed "THOP FLOPs
#   unavailable: No module named 'thop'"). try_compute_flops() already
#   had its own try/except fallback, so this changes nothing if thop
#   still cannot be installed (e.g. no internet in the Kaggle session).
# ================================================================

import os
import gc
import random
import time
import warnings
import re
import json
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------
# Optional: enable real FLOPs/MACs reporting if `thop` is missing.
# This does not change any modeling, data, or evaluation logic — it only
# lets the existing try_compute_flops() succeed instead of returning NaN.
# Safe no-op if there is no internet access in the Kaggle environment.
# ----------------------------------------------------------------
try:
    import thop  # noqa: F401
except ImportError:
    try:
        import subprocess
        import sys as _sys
        subprocess.run(
            [_sys.executable, "-m", "pip", "install", "-q", "thop"],
            check=False,
        )
    except Exception:
        pass

# ================================================================
# 1. CONFIGURATION
# ================================================================
SEED_SPLIT = 42
RUN_SEEDS = [42]  # Change to [42, 123, 2024] after the main run is stable.

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0
DROPOUT_P = 0.20

OPTIMIZER_NAME = "Adam"
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 10
SCHEDULER_FACTOR = 0.5
SCHEDULER_PATIENCE = 3
SCHEDULER_MIN_LR = 1e-7
N_FOLDS = 4

PRIMARY_AGGREGATION = "mean_probability"
SECONDARY_AGGREGATIONS = ["top10_mean_probability", "max_probability"]
SLICE_THRESHOLD = 0.50
FIXED_PATIENT_THRESHOLD = 0.50
HIGH_SPEC_TARGET = 0.90
TOP10_PERCENT = 0.10

N_BOOTSTRAP = 5000
BOOTSTRAP_SEED = 4242

LATENCY_WARMUP = 50
LATENCY_REPEATS_GPU = 200
LATENCY_REPEATS_CPU = 100

NORMALIZE_MEAN = [0.485, 0.456, 0.406]
NORMALIZE_STD = [0.229, 0.224, 0.225]

MODEL_NAMES = [
    "mobilenet_v2",
    "efficientnet_b0",
    "resnet50",
    "densenet121",
]

MODEL_LABELS = {
    "mobilenet_v2": "MobileNetV2",
    "efficientnet_b0": "EfficientNet-B0",
    "resnet50": "ResNet-50",
    "densenet121": "DenseNet-121",
}

MODEL_GROUPS = {
    "mobilenet_v2": "Lightweight",
    "efficientnet_b0": "Lightweight",
    "resnet50": "Larger CNN",
    "densenet121": "Larger CNN",
}

TEST_POSITIVE_PATIENTS = 4
TEST_NEGATIVE_PATIENTS = 5

FIG_DPI = 600

GRADCAM_ENABLED = True
GRADCAM_MAX_CASES_PER_MODEL = 6
GRADCAM_POSITIVE_QUANTILE = 0.93  # intentionally small/high-saliency region
GRADCAM_MIN_COMPONENT_PIXELS = 30

FAITHFULNESS_ENABLED = True
FAITHFULNESS_MAX_CASES_PER_MODEL = 12
FAITHFULNESS_FRACTIONS = np.linspace(0.0, 1.0, 21)
FAITHFULNESS_GRID_SIZE = 14
FAITHFULNESS_BLUR_KERNEL = 31

HEMORRHAGIC_DIR = "/kaggle/input/datasets/abdulkader90/brain-ct-hemorrhage-dataset/Data/Hemorrhagic/KANAMA"
NORMAL_DIR = "/kaggle/input/datasets/abdulkader90/brain-ct-hemorrhage-dataset/Data/NORMAL"

OUT_DIR = Path("/kaggle/working/ich_patient_level_publication_final")
OUT_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = Path("/kaggle/working/ICH_patient_level_publication_complete.zip")
INCLUDE_SOURCE_IMAGES_IN_ZIP = True

# Resolve the concrete CUDA device once (e.g. cuda:0) instead of using the
# generic cuda device string. This avoids false device-mismatch checks in
# recent PyTorch versions where torch.device("cuda") != torch.device("cuda:0").
if torch.cuda.is_available():
    DEVICE = torch.device("cuda", torch.cuda.current_device())
else:
    DEVICE = torch.device("cpu")

USE_AMP = DEVICE.type == "cuda"
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}

if DEVICE.type != "cuda":
    raise RuntimeError(
        "CUDA GPU is not available. In Kaggle, enable a GPU accelerator "
        "(for example NVIDIA T4) before running this notebook."
    )

# ================================================================
# 2. REPRODUCIBILITY / GPU
# ================================================================
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED_SPLIT)

GPU_NAME = torch.cuda.get_device_name(0)
print("=" * 120)
print("ICH PATIENT-LEVEL DEEP LEARNING — FINAL PUBLICATION PIPELINE — DEVICE-SAFE")
print("=" * 120)
print(f"CUDA available : {torch.cuda.is_available()}")
print(f"GPU            : {GPU_NAME}")
print(f"CUDA version   : {torch.version.cuda}")
print(f"PyTorch        : {torch.__version__}")
print(f"Models         : {[MODEL_LABELS[m] for m in MODEL_NAMES]}")
print(f"Seeds          : {RUN_SEEDS}")
print()

# ================================================================
# 3. DATA MANIFEST
# ================================================================
def natural_key(text):
    return [int(x) if x.isdigit() else x.lower() for x in re.split(r"(\d+)", str(text))]


def build_records(class_dir, label, class_name):
    if not os.path.isdir(class_dir):
        raise FileNotFoundError(f"Directory does not exist: {class_dir}")
    records = []
    folders = sorted(
        [f for f in os.listdir(class_dir) if os.path.isdir(os.path.join(class_dir, f))],
        key=natural_key,
    )
    for folder in folders:
        folder_path = os.path.join(class_dir, folder)
        files = sorted(
            [f for f in os.listdir(folder_path) if os.path.splitext(f)[1].lower() in VALID_EXTENSIONS],
            key=natural_key,
        )
        patient_id = f"{class_name}_{folder}"
        for fn in files:
            records.append(
                {
                    "patient_id": patient_id,
                    "folder_name": folder,
                    "class_name": class_name,
                    "label": int(label),
                    "image_path": os.path.join(folder_path, fn),
                }
            )
    return records


def build_manifest():
    df = pd.DataFrame(
        build_records(HEMORRHAGIC_DIR, 1, "HEMORRHAGIC")
        + build_records(NORMAL_DIR, 0, "NORMAL")
    )
    if df.empty:
        raise RuntimeError("Manifest is empty.")
    labels_per_patient = df.groupby("patient_id")["label"].nunique()
    if not (labels_per_patient == 1).all():
        raise RuntimeError("A patient has more than one label.")
    return df


df = build_manifest()

print("DATASET AUDIT")
print(f"Total slices          : {len(df)}")
print(f"Hemorrhagic slices    : {int((df.label == 1).sum())}")
print(f"Normal slices         : {int((df.label == 0).sum())}")
print(f"Total patients        : {df.patient_id.nunique()}")
print(f"Hemorrhagic patients  : {df.loc[df.label == 1, 'patient_id'].nunique()}")
print(f"Normal patients       : {df.loc[df.label == 0, 'patient_id'].nunique()}")
print(df.groupby("patient_id").size().describe())
print()

# ================================================================
# 4. FIXED OUTER PATIENT SPLIT + 4-FOLD CV
# ================================================================
patient_table = (
    df[["patient_id", "label"]]
    .drop_duplicates("patient_id")
    .sort_values("patient_id", key=lambda s: s.map(natural_key))
    .reset_index(drop=True)
)

pos = patient_table.loc[patient_table.label == 1, "patient_id"].tolist()
neg = patient_table.loc[patient_table.label == 0, "patient_id"].tolist()
if len(pos) != 18 or len(neg) != 27:
    raise RuntimeError(
        f"Expected 18 hemorrhagic + 27 normal patients; found {len(pos)} + {len(neg)}."
    )

rng = np.random.RandomState(SEED_SPLIT)
pos = np.array(pos, dtype=object)
neg = np.array(neg, dtype=object)
rng.shuffle(pos)
rng.shuffle(neg)

test_patients = set(pos[:TEST_POSITIVE_PATIENTS].tolist() + neg[:TEST_NEGATIVE_PATIENTS].tolist())
development_patients = set(pos[TEST_POSITIVE_PATIENTS:].tolist() + neg[TEST_NEGATIVE_PATIENTS:].tolist())

if len(test_patients) != 9 or len(development_patients) != 36:
    raise RuntimeError("Unexpected outer split size.")
if test_patients & development_patients:
    raise RuntimeError("Outer patient split overlap detected.")

df["outer_split"] = ""
df.loc[df.patient_id.isin(development_patients), "outer_split"] = "development"
df.loc[df.patient_id.isin(test_patients), "outer_split"] = "held_out_test"

# Fixed patient-level folds shared by all models and all seeds.
development_patient_table = patient_table[patient_table.patient_id.isin(development_patients)].copy().reset_index(drop=True)
development_patient_ids = development_patient_table.patient_id.to_numpy()
development_labels = development_patient_table.label.to_numpy()

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED_SPLIT)
fold_splits = list(skf.split(development_patient_ids, development_labels))

fold_assignments = []
df["cv_fold"] = np.nan
for fold, (tr_idx, va_idx) in enumerate(fold_splits, start=1):
    tr_patients = set(development_patient_ids[tr_idx])
    va_patients = set(development_patient_ids[va_idx])
    if len(tr_patients) != 27 or len(va_patients) != 9:
        raise RuntimeError(f"Fold {fold} does not contain 27/9 patients.")
    if tr_patients & va_patients:
        raise RuntimeError(f"Fold {fold} train/validation overlap.")
    if tr_patients & test_patients or va_patients & test_patients:
        raise RuntimeError(f"Fold {fold} overlaps held-out test patients.")
    for pid in va_patients:
        df.loc[df.patient_id == pid, "cv_fold"] = fold
    fold_assignments.append(
        {
            "fold": fold,
            "train_patients": 27,
            "validation_patients": 9,
            "train_ich_patients": int(sum(development_patient_table.loc[development_patient_table.patient_id.isin(tr_patients), "label"] == 1)),
            "train_normal_patients": int(sum(development_patient_table.loc[development_patient_table.patient_id.isin(tr_patients), "label"] == 0)),
            "validation_ich_patients": int(sum(development_patient_table.loc[development_patient_table.patient_id.isin(va_patients), "label"] == 1)),
            "validation_normal_patients": int(sum(development_patient_table.loc[development_patient_table.patient_id.isin(va_patients), "label"] == 0)),
        }
    )

if not df.loc[df.outer_split == "development", "cv_fold"].notna().all():
    raise RuntimeError("Not all development patients received a CV fold.")
if df.loc[df.outer_split == "held_out_test", "cv_fold"].notna().any():
    raise RuntimeError("Held-out test patients have CV folds.")

print("OUTER PATIENT SPLIT")
print(f"Development : {len(development_patients)} patients")
print(f"Held-out    : {len(test_patients)} patients")
print(f"Held-out IDs: {sorted(test_patients, key=natural_key)}")
print()

# ================================================================
# 5. TRANSFORMS
# ================================================================
train_transform = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=10),
        transforms.ColorJitter(brightness=0.15, contrast=0.15),
        transforms.RandomAffine(degrees=0, translate=(0.03, 0.03), scale=(0.95, 1.05)),
        transforms.ToTensor(),
        transforms.Normalize(mean=NORMALIZE_MEAN, std=NORMALIZE_STD),
    ]
)

val_transform = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=NORMALIZE_MEAN, std=NORMALIZE_STD),
    ]
)

# ================================================================
# 6. PATIENT-BALANCED LOSS
# ================================================================
def add_patient_balanced_weights(dataframe):
    out = dataframe.copy()
    patient_slice_counts = out.groupby("patient_id").size().to_dict()
    class_patient_counts = out.groupby("label")["patient_id"].nunique().to_dict()
    n_total = len(out)
    weights = []
    for _, row in out.iterrows():
        n_slices = patient_slice_counts[row.patient_id]
        n_class_patients = class_patient_counts[int(row.label)]
        # Each class receives 50% of total loss weight; each patient within a class
        # shares that class weight equally; slices within a patient share that patient weight.
        w = n_total * 0.5 / (n_class_patients * n_slices)
        weights.append(w)
    out["sample_weight"] = np.asarray(weights, dtype=np.float32)
    sums = out.groupby("label")["sample_weight"].sum()
    if not np.isclose(sums.iloc[0], sums.iloc[1], rtol=1e-5, atol=1e-5):
        raise RuntimeError("Patient-balanced class weights are not equal across classes.")
    return out

# ================================================================
# 7. DATASET / LOADER
# ================================================================
class BrainCTDataset(Dataset):
    def __init__(self, dataframe, transform=None, use_weights=False):
        self.df = dataframe.reset_index(drop=True).copy()
        self.transform = transform
        self.use_weights = use_weights

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        with Image.open(row.image_path) as im:
            image = im.convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        label = torch.tensor(float(row.label), dtype=torch.float32)
        weight = torch.tensor(
            float(row.sample_weight if self.use_weights else 1.0),
            dtype=torch.float32,
        )
        return image, label, weight, row.patient_id, row.image_path


def create_loader(dataframe, transform, shuffle, use_weights=False):
    return DataLoader(
        BrainCTDataset(dataframe, transform=transform, use_weights=use_weights),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        persistent_workers=False,
    )

# ================================================================
# 8. MODEL FACTORY
# ================================================================
def create_model(model_name):
    if model_name == "mobilenet_v2":
        model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 1)
    elif model_name == "efficientnet_b0":
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 1)
    elif model_name == "resnet50":
        model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        model.fc = nn.Sequential(
            nn.Dropout(DROPOUT_P),
            nn.Linear(model.fc.in_features, 1),
        )
    elif model_name == "densenet121":
        model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        model.classifier = nn.Sequential(
            nn.Dropout(DROPOUT_P),
            nn.Linear(model.classifier.in_features, 1),
        )
    else:
        raise ValueError(f"Unknown model: {model_name}")
    return model.to(DEVICE)


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def get_gradcam_target_layer(model, model_name):
    if model_name == "resnet50":
        return model.layer4[-1].conv3
    if model_name == "mobilenet_v2":
        return model.features[-1][0]
    if model_name == "efficientnet_b0":
        return model.features[-1][0]
    if model_name == "densenet121":
        return model.features.denseblock4.denselayer16.conv2
    raise ValueError(f"No Grad-CAM target for {model_name}")

# ================================================================
# 9. METRICS / AGGREGATION
# ================================================================
def calculate_metrics(labels, probabilities, threshold=0.5):
    labels = np.asarray(labels).astype(int)
    probabilities = np.asarray(probabilities).astype(float)
    predictions = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    if len(np.unique(labels)) == 2:
        auc = roc_auc_score(labels, probabilities)
        auprc = average_precision_score(labels, probabilities)
    else:
        auc = np.nan
        auprc = np.nan
    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions, zero_division=0),
        "sensitivity": recall_score(labels, predictions, zero_division=0),
        "specificity": tn / (tn + fp) if tn + fp else np.nan,
        "f1": f1_score(labels, predictions, zero_division=0),
        "auc": auc,
        "auprc": auprc,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def aggregate_patient_predictions(slice_df):
    rows = []
    for pid, g in slice_df.groupby("patient_id"):
        p = g.probability.to_numpy(dtype=float)
        k = max(1, int(np.ceil(len(p) * TOP10_PERCENT)))
        rows.append(
            {
                "patient_id": pid,
                "label": int(g.label.iloc[0]),
                "num_slices": len(p),
                "mean_probability": float(np.mean(p)),
                "top10_mean_probability": float(np.mean(np.sort(p)[-k:])),
                "max_probability": float(np.max(p)),
            }
        )
    return pd.DataFrame(rows)


def select_high_specificity_threshold(labels, probabilities, target_specificity=0.90):
    labels = np.asarray(labels).astype(int)
    probabilities = np.asarray(probabilities).astype(float)
    candidates = np.unique(np.concatenate(([0.0, 1.0], probabilities)))
    rows = []
    for threshold in candidates:
        m = calculate_metrics(labels, probabilities, float(threshold))
        rows.append(
            {
                "threshold": float(threshold),
                "sensitivity": m["sensitivity"],
                "specificity": m["specificity"],
                "accuracy": m["accuracy"],
                "precision": m["precision"],
                "f1": m["f1"],
            }
        )
    table = pd.DataFrame(rows)
    eligible = table[table.specificity >= target_specificity]
    fallback = False
    if len(eligible):
        chosen = eligible.sort_values(
            ["sensitivity", "specificity", "threshold"],
            ascending=[False, False, False],
        ).iloc[0]
    else:
        chosen = table.sort_values(
            ["specificity", "sensitivity", "threshold"],
            ascending=[False, False, False],
        ).iloc[0]
        fallback = True
    return float(chosen.threshold), fallback, table


def bootstrap_patient_level_ci(labels, probabilities, patient_ids, threshold=0.5, n_bootstrap=5000, seed=4242):
    labels = np.asarray(labels).astype(int)
    probabilities = np.asarray(probabilities).astype(float)
    patient_ids = np.asarray(patient_ids)
    unique_patients = np.unique(patient_ids)
    patient_indices = {pid: np.flatnonzero(patient_ids == pid) for pid in unique_patients}
    rng = np.random.default_rng(seed)
    names = ["accuracy", "precision", "sensitivity", "specificity", "f1", "auc", "auprc"]
    samples = {n: [] for n in names}
    for _ in range(n_bootstrap):
        chosen_patients = rng.choice(unique_patients, size=len(unique_patients), replace=True)
        idx = np.concatenate([patient_indices[pid] for pid in chosen_patients])
        y = labels[idx]
        p = probabilities[idx]
        if len(np.unique(y)) < 2:
            continue
        m = calculate_metrics(y, p, threshold)
        for n in names:
            if np.isfinite(m[n]):
                samples[n].append(m[n])
    ci = {}
    for n in names:
        arr = np.asarray(samples[n], dtype=float)
        ci[n] = (
            float(np.percentile(arr, 2.5)) if len(arr) else np.nan,
            float(np.percentile(arr, 97.5)) if len(arr) else np.nan,
        )
    return ci

# ================================================================
# 10. TRAINING
# ================================================================
def make_scaler():
    if not USE_AMP:
        return None
    try:
        return torch.amp.GradScaler("cuda", enabled=True)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=True)


def autocast_context():
    if USE_AMP:
        return torch.amp.autocast(device_type="cuda", enabled=True)
    return torch.autocast(device_type="cpu", enabled=False)


def patient_balanced_bce(logits, labels, weights):
    losses = F.binary_cross_entropy_with_logits(logits, labels, reduction="none")
    return (losses * weights).sum() / weights.sum().clamp_min(1e-8)


def _device_key(device):
    """Return a normalized (type, index) key for reliable device comparisons."""
    d = torch.device(device)
    if d.type == "cuda":
        return ("cuda", 0 if d.index is None else int(d.index))
    return (d.type, None)


def _assert_model_on_device(model, expected_device=DEVICE, context="model"):
    """Fail loudly on a real device mismatch, while treating cuda and cuda:0 equivalently."""
    try:
        param_device = next(model.parameters()).device
    except StopIteration as exc:
        raise RuntimeError(f"{context}: model has no parameters.") from exc

    if _device_key(param_device) != _device_key(expected_device):
        raise RuntimeError(
            f"{context}: model is on {param_device}, but expected {expected_device}. "
            "A previous CPU/GPU operation may have moved the model."
        )


def _prepare_model_for_device(model, device=DEVICE, context="model"):
    """Move to the requested device, enforce FP32 storage, then verify placement."""
    target = torch.device(device)
    model = model.to(target)
    model.float()
    _assert_model_on_device(model, target, context)
    return model


def train_one_epoch(model, loader, optimizer, scaler):
    model = _prepare_model_for_device(model, DEVICE, "train_one_epoch")
    model.train()
    total_loss = 0.0
    y_all, p_all = [], []

    for images, labels, weights, _, _ in loader:
        images = images.to(DEVICE, dtype=torch.float32, non_blocking=True)
        labels = labels.to(DEVICE, dtype=torch.float32, non_blocking=True)
        weights = weights.to(DEVICE, dtype=torch.float32, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast_context():
            logits = model(images).reshape(-1)
            loss = patient_balanced_bce(logits, labels, weights)

        if USE_AMP:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        total_loss += float(loss.item()) * images.size(0)
        y_all.extend(labels.detach().cpu().numpy().tolist())
        p_all.extend(torch.sigmoid(logits).detach().float().cpu().numpy().tolist())

    return total_loss / len(loader.dataset), calculate_metrics(y_all, p_all, SLICE_THRESHOLD)


def evaluate_model(model, loader):
    """Evaluate in FP32 on the configured DEVICE.

    Evaluation intentionally disables AMP. This eliminates mixed-dtype ambiguity
    at the final validation/test stage while training can still use AMP for speed.
    """
    model = _prepare_model_for_device(model, DEVICE, "evaluate_model")
    model.eval()
    ys, ps, pids, paths = [], [], [], []
    total_loss = 0.0

    with torch.inference_mode():
        for images, labels, _, patient_ids, image_paths in loader:
            images = images.to(DEVICE, dtype=torch.float32, non_blocking=True)
            labels = labels.to(DEVICE, dtype=torch.float32, non_blocking=True)

            logits = model(images).reshape(-1)
            loss = F.binary_cross_entropy_with_logits(logits, labels)
            probabilities = torch.sigmoid(logits)

            total_loss += float(loss.item()) * images.size(0)
            ys.extend(labels.cpu().numpy().tolist())
            ps.extend(probabilities.float().cpu().numpy().tolist())
            pids.extend(list(patient_ids))
            paths.extend(list(image_paths))

    slice_df = pd.DataFrame(
        {
            "patient_id": pids,
            "label": np.asarray(ys).astype(int),
            "probability": np.asarray(ps).astype(float),
            "image_path": paths,
        }
    )
    if slice_df.empty:
        raise RuntimeError("evaluate_model: loader produced zero samples.")

    patient_df = aggregate_patient_predictions(slice_df)
    if patient_df.empty:
        raise RuntimeError("evaluate_model: patient aggregation produced zero patients.")

    return {
        "loss": total_loss / len(loader.dataset),
        "slice_df": slice_df,
        "slice_metrics": calculate_metrics(slice_df.label, slice_df.probability, SLICE_THRESHOLD),
        "patient_df": patient_df,
        "patient_metrics_mean": calculate_metrics(patient_df.label, patient_df.mean_probability, FIXED_PATIENT_THRESHOLD),
        "patient_metrics_top10": calculate_metrics(patient_df.label, patient_df.top10_mean_probability, FIXED_PATIENT_THRESHOLD),
        "patient_metrics_max": calculate_metrics(patient_df.label, patient_df.max_probability, FIXED_PATIENT_THRESHOLD),
    }


def build_optimizer(model):
    if OPTIMIZER_NAME.lower() == "adam":
        return torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    if OPTIMIZER_NAME.lower() == "adamw":
        return torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    raise ValueError("Unsupported optimizer.")

# ================================================================
# 11. TRAIN ONE FOLD
# ================================================================
def train_one_fold(model_name, seed, fold_number, train_df, val_df):
    weighted_train = add_patient_balanced_weights(train_df)
    train_loader = create_loader(weighted_train, train_transform, True, True)
    val_loader = create_loader(val_df, val_transform, False, False)
    seed_everything(seed)
    model = create_model(model_name)
    total_params, trainable_params = count_parameters(model)
    optimizer = build_optimizer(model)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=SCHEDULER_FACTOR,
        patience=SCHEDULER_PATIENCE,
        min_lr=SCHEDULER_MIN_LR,
    )
    scaler = make_scaler()
    best_auc = -np.inf
    best_epoch = 0
    best_state = None
    no_improve = 0
    history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        t0 = time.time()
        train_loss, train_metrics = train_one_epoch(model, train_loader, optimizer, scaler)
        val = evaluate_model(model, val_loader)
        val_auc = val["patient_metrics_mean"]["auc"]
        scheduler.step(val_auc if np.isfinite(val_auc) else -np.inf)
        lr = optimizer.param_groups[0]["lr"]
        history.append(
            {
                "record_type": "cv_epoch",
                "model": model_name,
                "model_group": MODEL_GROUPS[model_name],
                "seed": seed,
                "fold": fold_number,
                "epoch": epoch,
                "train_loss": train_loss,
                "train_auc": train_metrics["auc"],
                "val_patient_auc": val_auc,
                "val_patient_auprc": val["patient_metrics_mean"]["auprc"],
                "learning_rate": lr,
                "epoch_seconds": time.time() - t0,
            }
        )
        print(
            f"{MODEL_LABELS[model_name]:16s} | fold {fold_number} | epoch {epoch:03d} | "
            f"val patient AUROC={val_auc:.4f} | AUPRC={val['patient_metrics_mean']['auprc']:.4f}"
        )
        if np.isfinite(val_auc) and val_auc > best_auc:
            best_auc = val_auc
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= EARLY_STOPPING_PATIENCE:
            break

    if best_state is None:
        raise RuntimeError(f"No valid best state for {model_name}, fold {fold_number}.")

    model.load_state_dict(best_state)
    final_val = evaluate_model(model, val_loader)
    val_patient = final_val["patient_df"].copy()
    val_patient["model"] = model_name
    val_patient["model_group"] = MODEL_GROUPS[model_name]
    val_patient["seed"] = seed
    val_patient["fold"] = fold_number

    result = {
        "model": model_name,
        "model_group": MODEL_GROUPS[model_name],
        "seed": seed,
        "fold": fold_number,
        "best_epoch": best_epoch,
        "best_val_patient_auc": best_auc,
        "val": final_val,
        "val_patient": val_patient,
        "history": pd.DataFrame(history),
        "total_params": total_params,
        "trainable_params": trainable_params,
    }
    del model, optimizer, scheduler, scaler, train_loader, val_loader
    gc.collect()
    torch.cuda.empty_cache()
    return result

# ================================================================
# 12. FINAL MODEL
# ================================================================
def train_final_model(model_name, seed, development_df, epochs):
    weighted_train = add_patient_balanced_weights(development_df)
    loader = create_loader(weighted_train, train_transform, True, True)
    seed_everything(seed)
    model = create_model(model_name)
    optimizer = build_optimizer(model)
    scaler = make_scaler()
    history = []
    start = time.time()
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        loss, metrics = train_one_epoch(model, loader, optimizer, scaler)
        history.append(
            {
                "record_type": "final_epoch",
                "model": model_name,
                "model_group": MODEL_GROUPS[model_name],
                "seed": seed,
                "epoch": epoch,
                "train_loss": loss,
                "train_auc": metrics["auc"],
                "learning_rate": optimizer.param_groups[0]["lr"],
                "epoch_seconds": time.time() - t0,
            }
        )
        print(
            f"FINAL {MODEL_LABELS[model_name]:16s} | epoch {epoch:03d}/{epochs} | "
            f"train AUC={metrics['auc']:.4f}"
        )
    total_seconds = time.time() - start
    return model, pd.DataFrame(history), total_seconds

# ================================================================
# 13. EFFICIENCY
# ================================================================
def measure_latency(model, device, repeats, warmup):
    """Measure latency without permanently changing the caller's model device."""
    if device.type == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("Requested CUDA latency measurement, but CUDA is unavailable.")

    original_device = next(model.parameters()).device
    original_training = model.training

    try:
        model.to(device).float().eval()
        dummy = torch.randn(
            1,
            3,
            IMG_SIZE,
            IMG_SIZE,
            device=device,
            dtype=torch.float32,
        )

        with torch.inference_mode():
            for _ in range(int(warmup)):
                _ = model(dummy)

            if device.type == "cuda":
                torch.cuda.synchronize(device)

            timings = []
            for _ in range(int(repeats)):
                if device.type == "cuda":
                    torch.cuda.synchronize(device)
                t0 = time.perf_counter()
                _ = model(dummy)
                if device.type == "cuda":
                    torch.cuda.synchronize(device)
                timings.append((time.perf_counter() - t0) * 1000.0)

        arr = np.asarray(timings, dtype=float)
        return {
            "mean_ms": float(arr.mean()),
            "median_ms": float(np.median(arr)),
            "sd_ms": float(arr.std(ddof=1)) if len(arr) > 1 else 0.0,
            "p95_ms": float(np.percentile(arr, 95)),
        }
    finally:
        # CRITICAL: restore the exact device from which the benchmark started.
        model.to(original_device).float()
        model.train(original_training)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def try_compute_flops(model):
    try:
        from thop import profile
        _assert_model_on_device(model, DEVICE, "try_compute_flops")
        model = model.float().eval()
        dummy = torch.randn(
            1,
            3,
            IMG_SIZE,
            IMG_SIZE,
            device=DEVICE,
            dtype=torch.float32,
        )
        with torch.inference_mode():
            macs, params = profile(model, inputs=(dummy,), verbose=False)
        return float(2 * macs), float(macs), float(params)
    except Exception as exc:
        print(f"THOP FLOPs unavailable: {exc}")
        return np.nan, np.nan, np.nan

# ================================================================
# 14. ROBUST GRAD-CAM
# ================================================================
def generate_gradcam(model, model_name, image_tensor):
    """Robust FP32 Grad-CAM using an activation-tensor gradient hook.

    This does not use register_backward_hook/register_full_backward_hook.
    The positive class logit is the target, so the map explains the ICH score.
    """
    model = _prepare_model_for_device(model, DEVICE, "Grad-CAM")
    model.eval()

    if image_tensor.ndim == 3:
        x = image_tensor.unsqueeze(0)
    elif image_tensor.ndim == 4 and image_tensor.shape[0] == 1:
        x = image_tensor
    else:
        raise ValueError(
            f"Grad-CAM expects [3,H,W] or [1,3,H,W], got {tuple(image_tensor.shape)}"
        )

    x = x.detach().clone().to(DEVICE, dtype=torch.float32)
    x.requires_grad_(True)

    target_layer = get_gradcam_target_layer(model, model_name)
    cache = {}

    def forward_hook(module, inputs, output):
        if not isinstance(output, torch.Tensor):
            raise RuntimeError(
                f"Grad-CAM target output is not a tensor for {model_name}."
            )
        cache["activation"] = output.detach()
        output.register_hook(
            lambda grad: cache.__setitem__("gradient", grad.detach())
        )

    handle = target_layer.register_forward_hook(forward_hook)

    try:
        model.zero_grad(set_to_none=True)

        # Explicit FP32 forward; do not inherit the training AMP context.
        with torch.autocast(device_type="cuda", enabled=False):
            logits = model(x).reshape(-1)
            if logits.numel() != 1:
                raise RuntimeError(
                    f"Grad-CAM expected exactly one logit for {model_name}; "
                    f"got shape {tuple(logits.shape)}"
                )
            score = logits[0]

        probability = float(torch.sigmoid(score.detach()).cpu().item())
        score.backward()

        if "activation" not in cache or "gradient" not in cache:
            raise RuntimeError(
                f"Grad-CAM activation/gradient was not captured for {model_name}."
            )

        activation = cache["activation"]
        gradient = cache["gradient"]

        if activation.ndim != 4 or gradient.shape != activation.shape:
            raise RuntimeError(
                f"Grad-CAM tensor mismatch for {model_name}: "
                f"activation={tuple(activation.shape)}, "
                f"gradient={tuple(gradient.shape)}"
            )

        weights = gradient.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * activation).sum(dim=1, keepdim=True))
        cam = F.interpolate(
            cam,
            size=(IMG_SIZE, IMG_SIZE),
            mode="bilinear",
            align_corners=False,
        )[0, 0]

        cam = cam.detach().cpu().numpy().astype(np.float32)
        if not np.isfinite(cam).all():
            raise RuntimeError(f"Grad-CAM produced NaN/Inf for {model_name}.")

        cam -= cam.min()
        maximum = float(cam.max())
        if maximum > 1e-12:
            cam /= maximum
        else:
            cam.fill(0.0)

        return np.clip(cam, 0.0, 1.0), probability

    finally:
        handle.remove()
        model.zero_grad(set_to_none=True)

def sparse_gradcam_mask(cam, quantile=0.93, min_component_pixels=30):
    from scipy import ndimage
    cam = np.asarray(cam, dtype=np.float32)
    positive = cam[cam > 0]
    if positive.size == 0:
        return np.zeros_like(cam, dtype=bool)
    threshold = float(np.quantile(positive, quantile))
    binary = cam >= threshold
    labeled, n = ndimage.label(binary)
    if n == 0:
        return np.zeros_like(cam, dtype=bool)
    sizes = np.bincount(labeled.ravel())
    keep_ids = [i for i in range(1, n + 1) if sizes[i] >= min_component_pixels]
    if not keep_ids:
        # Keep only the strongest 1% if connected-component filtering removes everything.
        k = max(20, int(cam.size * 0.01))
        idx = np.argpartition(cam.ravel(), -k)[-k:]
        mask = np.zeros_like(cam, dtype=bool)
        mask.ravel()[idx] = True
        return mask
    return np.isin(labeled, keep_ids)


def save_gradcam_figure(display_image, cam, model_label, true_label, probability, out_png, out_pdf):
    mask = sparse_gradcam_mask(cam, GRADCAM_POSITIVE_QUANTILE, GRADCAM_MIN_COMPONENT_PIXELS)
    masked_cam = np.ma.masked_where(~mask, cam)
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(display_image)
    axes[0].set_title("Original CT")
    axes[0].axis("off")
    axes[1].imshow(display_image)
    axes[1].imshow(masked_cam, cmap="turbo", alpha=0.60, vmin=0, vmax=1)
    axes[1].set_title("Sparse high-saliency Grad-CAM")
    axes[1].axis("off")
    axes[2].imshow(display_image)
    axes[2].imshow(masked_cam, cmap="turbo", alpha=0.48, vmin=0, vmax=1)
    axes[2].set_title("Overlay")
    axes[2].axis("off")
    fig.suptitle(
        f"{model_label} | True={true_label} | P(ICH)={probability:.3f}",
        fontsize=14,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.93])
    fig.savefig(out_png, dpi=FIG_DPI, bbox_inches="tight", facecolor="white")
    fig.savefig(out_pdf, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return float(mask.mean())

# ================================================================
# 15. GRAD-CAM FAITHFULNESS
# ================================================================
def unnormalize(image_tensor):
    mean = torch.tensor(NORMALIZE_MEAN, device=image_tensor.device, dtype=image_tensor.dtype).view(1, 3, 1, 1)
    std = torch.tensor(NORMALIZE_STD, device=image_tensor.device, dtype=image_tensor.dtype).view(1, 3, 1, 1)
    return image_tensor * std + mean


def normalize_raw(image_tensor):
    mean = torch.tensor(NORMALIZE_MEAN, device=image_tensor.device, dtype=image_tensor.dtype).view(1, 3, 1, 1)
    std = torch.tensor(NORMALIZE_STD, device=image_tensor.device, dtype=image_tensor.dtype).view(1, 3, 1, 1)
    return (image_tensor - mean) / std


def blur_baseline(raw_tensor):
    return F.avg_pool2d(
        raw_tensor,
        kernel_size=FAITHFULNESS_BLUR_KERNEL,
        stride=1,
        padding=FAITHFULNESS_BLUR_KERNEL // 2,
    ).clamp(0, 1)


def faithfulness_curves(model, image_tensor, cam, target_class):
    """Deletion/insertion using a blurred RGB baseline.

    Fractions refer to the fraction of spatial CAM cells modified/restored.
    target_class=1 evaluates ICH confidence; target_class=0 evaluates normal confidence.
    """
    model.eval()
    x = image_tensor.to(DEVICE, dtype=torch.float32)
    raw = unnormalize(x).clamp(0, 1)
    baseline = blur_baseline(raw)
    cam_t = torch.from_numpy(cam).float()
    h, w = cam.shape
    cell_h = int(np.ceil(h / FAITHFULNESS_GRID_SIZE))
    cell_w = int(np.ceil(w / FAITHFULNESS_GRID_SIZE))
    cells = []
    for gy in range(FAITHFULNESS_GRID_SIZE):
        for gx in range(FAITHFULNESS_GRID_SIZE):
            y0, y1 = gy * cell_h, min((gy + 1) * cell_h, h)
            x0, x1 = gx * cell_w, min((gx + 1) * cell_w, w)
            if y0 < h and x0 < w:
                cells.append((float(cam_t[y0:y1, x0:x1].mean()), y0, y1, x0, x1))
    cells.sort(key=lambda z: z[0], reverse=True)

    def confidence(inp):
        norm = normalize_raw(inp)
        with torch.inference_mode():
            prob_ich = torch.sigmoid(model(norm).reshape(-1)[0]).item()
        return float(prob_ich if target_class == 1 else 1.0 - prob_ich)

    deletion_scores = []
    insertion_scores = []
    total_cells = len(cells)

    for frac in FAITHFULNESS_FRACTIONS:
        k = int(round(float(frac) * total_cells))
        deletion = raw.clone()
        insertion = baseline.clone()
        for _, y0, y1, x0, x1 in cells[:k]:
            deletion[:, :, y0:y1, x0:x1] = baseline[:, :, y0:y1, x0:x1]
            insertion[:, :, y0:y1, x0:x1] = raw[:, :, y0:y1, x0:x1]
        deletion_scores.append(confidence(deletion))
        insertion_scores.append(confidence(insertion))

    xvals = FAITHFULNESS_FRACTIONS
    if hasattr(np, "trapezoid"):
        d_auc = float(np.trapezoid(deletion_scores, xvals))
        i_auc = float(np.trapezoid(insertion_scores, xvals))
    else:
        d_auc = float(np.trapz(deletion_scores, xvals))
        i_auc = float(np.trapz(insertion_scores, xvals))
    return np.asarray(deletion_scores), np.asarray(insertion_scores), d_auc, i_auc

# ================================================================
# 15B. FAIL-FAST PREFLIGHT
# ================================================================
def run_preflight():
    """Run cheap real-model checks before the expensive CV loop."""
    print("\n" + "=" * 120)
    print("FAIL-FAST PREFLIGHT")
    print("=" * 120)

    if DEVICE.type != "cuda":
        raise RuntimeError(
            "Preflight failed: CUDA is unavailable. In Kaggle, select a GPU accelerator "
            "before starting this notebook."
        )

    sample_path = str(df.iloc[0].image_path)
    with Image.open(sample_path) as im:
        sample_pil = im.convert("RGB")
    sample_tensor = val_transform(sample_pil)

    # Explicitly test the exact failure mode that previously wasted the run:
    # a CPU latency benchmark must restore the model to CUDA before evaluation.
    restoration_test_model = create_model("mobilenet_v2")
    try:
        _prepare_model_for_device(restoration_test_model, DEVICE, "preflight/device_restore/before")
        before_key = _device_key(next(restoration_test_model.parameters()).device)
        cpu_stats = measure_latency(restoration_test_model, torch.device("cpu"), repeats=1, warmup=0)
        after_key = _device_key(next(restoration_test_model.parameters()).device)
        if before_key != after_key or after_key != _device_key(DEVICE):
            raise RuntimeError(
                "Device-restore preflight failed: model did not return to the CUDA device "
                "after the CPU latency test."
            )
        print(
            "Preflight: CPU-latency device restoration PASS | "
            f"CPU smoke latency={cpu_stats['mean_ms']:.2f} ms | restored={after_key}"
        )
    finally:
        del restoration_test_model
        gc.collect()
        torch.cuda.empty_cache()

    # Validate all four architectures, including their Grad-CAM targets.
    for model_name in MODEL_NAMES:
        print(f"Preflight: {MODEL_LABELS[model_name]}")
        model = None
        try:
            seed_everything(SEED_SPLIT)
            model = create_model(model_name)
            _prepare_model_for_device(model, DEVICE, f"preflight/{model_name}")

            with torch.inference_mode():
                output = model(
                    sample_tensor.unsqueeze(0).to(
                        DEVICE,
                        dtype=torch.float32,
                    )
                )
            if tuple(output.shape) != (1, 1):
                raise RuntimeError(
                    f"Unexpected output shape for {model_name}: {tuple(output.shape)}"
                )

            cam, probability = generate_gradcam(
                model,
                model_name,
                sample_tensor,
            )
            if cam.shape != (IMG_SIZE, IMG_SIZE):
                raise RuntimeError(
                    f"Unexpected Grad-CAM shape for {model_name}: {cam.shape}"
                )
            if not np.isfinite(cam).all() or not np.isfinite(probability):
                raise RuntimeError(f"Non-finite preflight output for {model_name}.")

            print(
                f"  PASS | forward={tuple(output.shape)} | "
                f"Grad-CAM={cam.shape} | P(ICH)={probability:.4f}"
            )
        finally:
            del model
            gc.collect()
            torch.cuda.empty_cache()

    # Verify split dimensions before training.
    development_patient_count = df.loc[
        df.outer_split == "development", "patient_id"
    ].nunique()
    test_patient_count = df.loc[
        df.outer_split == "held_out_test", "patient_id"
    ].nunique()
    if development_patient_count != 36 or test_patient_count != 9:
        raise RuntimeError(
            f"Unexpected patient split: development={development_patient_count}, "
            f"held_out_test={test_patient_count}; expected 36 and 9."
        )

    print("PREFLIGHT PASSED: device, four model forwards, Grad-CAM targets, and split checks.")
    print("No expensive model training begins before these checks pass.")

run_preflight()

# ================================================================
# 16. MASTER OUTPUT STORE
# ================================================================
master_rows = []

def add_master(record_type, **kwargs):
    row = {"record_type": record_type}
    row.update(kwargs)
    master_rows.append(row)

# Save manifests to master rows later.
for row in fold_assignments:
    add_master("fold_assignment", **row)
for _, row in df.iterrows():
    add_master(
        "manifest",
        patient_id=row.patient_id,
        class_name=row.class_name,
        label=int(row.label),
        image_path=row.image_path,
        outer_split=row.outer_split,
        cv_fold=row.cv_fold if pd.notna(row.cv_fold) else np.nan,
    )

# ================================================================
# 17. MAIN EXPERIMENT
# ================================================================
development_df = df[df.outer_split == "development"].reset_index(drop=True)
held_out_test_df = df[df.outer_split == "held_out_test"].reset_index(drop=True)

all_test_patient_tables = []
all_oof_patient_tables = []
all_efficiency_tables = []
all_gradcam_case_tables = []
all_faithfulness_case_tables = []

for model_name in MODEL_NAMES:
    for seed in RUN_SEEDS:
        print("\n" + "=" * 120)
        print(f"MODEL: {MODEL_LABELS[model_name]} | SEED: {seed}")
        print("=" * 120)
        seed_everything(seed)

        cv_results = []
        for fold_number, (tr_idx, va_idx) in enumerate(fold_splits, start=1):
            train_patients = set(development_patient_ids[tr_idx])
            val_patients = set(development_patient_ids[va_idx])
            fold_train = df[df.patient_id.isin(train_patients)].reset_index(drop=True)
            fold_val = df[df.patient_id.isin(val_patients)].reset_index(drop=True)
            result = train_one_fold(model_name, seed, fold_number, fold_train, fold_val)
            cv_results.append(result)

            # Fold summary.
            add_master(
                "cv_fold_summary",
                model=model_name,
                model_label=MODEL_LABELS[model_name],
                model_group=MODEL_GROUPS[model_name],
                seed=seed,
                fold=fold_number,
                train_patients=27,
                validation_patients=9,
                best_epoch=result["best_epoch"],
                best_validation_patient_auc=result["best_val_patient_auc"],
                validation_patient_auc=result["val"]["patient_metrics_mean"]["auc"],
                validation_patient_auprc=result["val"]["patient_metrics_mean"]["auprc"],
                validation_patient_sensitivity=result["val"]["patient_metrics_mean"]["sensitivity"],
                validation_patient_specificity=result["val"]["patient_metrics_mean"]["specificity"],
                validation_patient_f1=result["val"]["patient_metrics_mean"]["f1"],
            )
            for _, hrow in result["history"].iterrows():
                add_master(**hrow.to_dict())

        # OOF patient table: each of 36 development patients appears exactly once.
        oof_patient = pd.concat([r["val_patient"] for r in cv_results], ignore_index=True)
        if len(oof_patient) != 36 or oof_patient.patient_id.nunique() != 36:
            raise RuntimeError(f"OOF patient count mismatch for {model_name}.")
        if oof_patient.patient_id.duplicated().any():
            raise RuntimeError(f"Duplicate OOF patients for {model_name}.")
        all_oof_patient_tables.append(oof_patient)

        oof_mean = calculate_metrics(oof_patient.label, oof_patient.mean_probability, FIXED_PATIENT_THRESHOLD)
        threshold, fallback, threshold_candidates = select_high_specificity_threshold(
            oof_patient.label,
            oof_patient.mean_probability,
            HIGH_SPEC_TARGET,
        )
        for _, row in oof_patient.iterrows():
            add_master(
                "oof_patient_prediction",
                model=model_name,
                model_label=MODEL_LABELS[model_name],
                model_group=MODEL_GROUPS[model_name],
                seed=seed,
                patient_id=row.patient_id,
                label=int(row.label),
                num_slices=row.num_slices,
                mean_probability=row.mean_probability,
                top10_mean_probability=row.top10_mean_probability,
                max_probability=row.max_probability,
            )
        add_master(
            "oof_summary",
            model=model_name,
            model_label=MODEL_LABELS[model_name],
            model_group=MODEL_GROUPS[model_name],
            seed=seed,
            patients=36,
            oof_patient_auc=oof_mean["auc"],
            oof_patient_auprc=oof_mean["auprc"],
            oof_patient_accuracy=oof_mean["accuracy"],
            oof_patient_sensitivity=oof_mean["sensitivity"],
            oof_patient_specificity=oof_mean["specificity"],
            oof_patient_f1=oof_mean["f1"],
            high_specificity_target=HIGH_SPEC_TARGET,
            high_specificity_threshold=threshold,
            threshold_fallback=fallback,
        )

        best_epochs = [r["best_epoch"] for r in cv_results]
        final_epochs = max(1, min(int(np.median(best_epochs)), MAX_EPOCHS))
        add_master(
            "model_selection",
            model=model_name,
            model_label=MODEL_LABELS[model_name],
            model_group=MODEL_GROUPS[model_name],
            seed=seed,
            mean_cv_best_epoch=float(np.mean(best_epochs)),
            median_cv_best_epoch=int(final_epochs),
            mean_cv_auc=float(np.mean([r["best_val_patient_auc"] for r in cv_results])),
            sd_cv_auc=float(np.std([r["best_val_patient_auc"] for r in cv_results], ddof=1)),
        )

        # Final model on all 36 development patients.
        final_model, final_history, final_seconds = train_final_model(
            model_name, seed, development_df, final_epochs
        )
        for _, hrow in final_history.iterrows():
            add_master(**hrow.to_dict())

        checkpoint_path = OUT_DIR / f"{model_name}_seed{seed}_final.pth"
        torch.save(
            {
                "model_name": model_name,
                "model_group": MODEL_GROUPS[model_name],
                "seed": seed,
                "split_seed": SEED_SPLIT,
                "development_patients": 36,
                "held_out_test_patients": 9,
                "cv_folds": 4,
                "cv_best_epochs": best_epochs,
                "final_training_epochs": final_epochs,
                "img_size": IMG_SIZE,
                "batch_size": BATCH_SIZE,
                "normalization_mean": NORMALIZE_MEAN,
                "normalization_std": NORMALIZE_STD,
                "optimizer": OPTIMIZER_NAME,
                "learning_rate": LEARNING_RATE,
                "weight_decay": WEIGHT_DECAY,
                "dropout": DROPOUT_P,
                "primary_aggregation": PRIMARY_AGGREGATION,
                "fixed_patient_threshold": FIXED_PATIENT_THRESHOLD,
                "high_specificity_threshold_from_oof": threshold,
                "patient_balanced_loss": True,
                "model_state_dict": final_model.state_dict(),
            },
            checkpoint_path,
        )

        # Efficiency.
        total_params, trainable_params = count_parameters(final_model)
        gpu_latency = measure_latency(final_model, DEVICE, LATENCY_REPEATS_GPU, LATENCY_WARMUP)
        cpu_latency = None
        try:
            cpu_latency = measure_latency(final_model, torch.device("cpu"), LATENCY_REPEATS_CPU, 20)
        except Exception as exc:
            print(f"CPU latency not available for {model_name}: {exc}")
            cpu_latency = None
        flops, macs, thop_params = try_compute_flops(final_model)
        # Never allow any benchmarking helper to strand the final model on another device.
        final_model = _prepare_model_for_device(final_model, DEVICE, "after_efficiency")
        model_size_mb = checkpoint_path.stat().st_size / (1024 ** 2)
        # FIX: measure_latency() returns a dict of stats (mean/median/sd/p95).
        # Storing that dict directly in a single DataFrame cell (as the
        # original script did) breaks any later groupby(...).mean() call —
        # this was the exact cause of the TypeError that ended the previous
        # run right after DenseNet-121 finished. Each stat is now stored in
        # its own scalar column instead. No other value here is changed.
        efficiency_row = {
            "record_type": "efficiency",
            "model": model_name,
            "model_label": MODEL_LABELS[model_name],
            "model_group": MODEL_GROUPS[model_name],
            "seed": seed,
            "total_parameters": total_params,
            "trainable_parameters": trainable_params,
            "parameters_million": total_params / 1e6,
            "model_size_mb": model_size_mb,
            "gpu_latency_mean_ms": gpu_latency["mean_ms"],
            "gpu_latency_median_ms": gpu_latency["median_ms"],
            "gpu_latency_sd_ms": gpu_latency["sd_ms"],
            "gpu_latency_p95_ms": gpu_latency["p95_ms"],
            "cpu_latency_mean_ms": cpu_latency["mean_ms"] if cpu_latency is not None else np.nan,
            "cpu_latency_median_ms": cpu_latency["median_ms"] if cpu_latency is not None else np.nan,
            "cpu_latency_sd_ms": cpu_latency["sd_ms"] if cpu_latency is not None else np.nan,
            "cpu_latency_p95_ms": cpu_latency["p95_ms"] if cpu_latency is not None else np.nan,
            "FLOPs": flops,
            "MACs": macs,
            "final_training_seconds": final_seconds,
        }
        all_efficiency_tables.append(pd.DataFrame([efficiency_row]))
        add_master(**efficiency_row)

        # Test evaluation ONCE for this fixed final model.
        final_model = _prepare_model_for_device(final_model, DEVICE, "before_heldout_test")
        test_loader = create_loader(held_out_test_df, val_transform, False, False)
        test_eval = evaluate_model(final_model, test_loader)
        test_slice = test_eval["slice_df"].copy()
        test_slice["model"] = model_name
        test_slice["seed"] = seed
        test_slice["split"] = "held_out_test"
        test_patient = test_eval["patient_df"].copy()
        test_patient["model"] = model_name
        test_patient["model_group"] = MODEL_GROUPS[model_name]
        test_patient["seed"] = seed
        test_patient["prediction_mean_050"] = (test_patient.mean_probability >= FIXED_PATIENT_THRESHOLD).astype(int)
        test_patient["prediction_top10_050"] = (test_patient.top10_mean_probability >= FIXED_PATIENT_THRESHOLD).astype(int)
        test_patient["prediction_max_050"] = (test_patient.max_probability >= FIXED_PATIENT_THRESHOLD).astype(int)
        test_patient["oof_high_specificity_threshold"] = threshold
        test_patient["prediction_high_specificity"] = (test_patient.mean_probability >= threshold).astype(int)
        all_test_patient_tables.append(test_patient)

        test_metrics = calculate_metrics(
            test_patient.label,
            test_patient.mean_probability,
            FIXED_PATIENT_THRESHOLD,
        )
        test_ci = bootstrap_patient_level_ci(
            test_patient.label,
            test_patient.mean_probability,
            test_patient.patient_id,
            FIXED_PATIENT_THRESHOLD,
            N_BOOTSTRAP,
            BOOTSTRAP_SEED + seed,
        )
        add_master(
            "heldout_test_summary",
            model=model_name,
            model_label=MODEL_LABELS[model_name],
            model_group=MODEL_GROUPS[model_name],
            seed=seed,
            test_patients=len(test_patient),
            test_slices=len(test_slice),
            aggregation=PRIMARY_AGGREGATION,
            threshold=FIXED_PATIENT_THRESHOLD,
            accuracy=test_metrics["accuracy"],
            sensitivity=test_metrics["sensitivity"],
            specificity=test_metrics["specificity"],
            precision=test_metrics["precision"],
            f1=test_metrics["f1"],
            auc=test_metrics["auc"],
            auprc=test_metrics["auprc"],
            auc_ci_low=test_ci["auc"][0],
            auc_ci_high=test_ci["auc"][1],
            auprc_ci_low=test_ci["auprc"][0],
            auprc_ci_high=test_ci["auprc"][1],
            tn=test_metrics["tn"],
            fp=test_metrics["fp"],
            fn=test_metrics["fn"],
            tp=test_metrics["tp"],
        )
        for _, row in test_patient.iterrows():
            add_master(
                "heldout_test_patient_prediction",
                model=model_name,
                model_label=MODEL_LABELS[model_name],
                model_group=MODEL_GROUPS[model_name],
                seed=seed,
                patient_id=row.patient_id,
                label=int(row.label),
                num_slices=row.num_slices,
                mean_probability=row.mean_probability,
                top10_mean_probability=row.top10_mean_probability,
                max_probability=row.max_probability,
                prediction_mean_050=int(row.prediction_mean_050),
                prediction_top10_050=int(row.prediction_top10_050),
                prediction_max_050=int(row.prediction_max_050),
                oof_high_specificity_threshold=threshold,
                prediction_high_specificity=int(row.prediction_high_specificity),
            )

        for _, row in test_slice.iterrows():
            add_master(
                "heldout_test_slice_prediction",
                model=model_name,
                model_label=MODEL_LABELS[model_name],
                model_group=MODEL_GROUPS[model_name],
                seed=seed,
                patient_id=row.patient_id,
                label=int(row.label),
                probability=row.probability,
                image_path=row.image_path,
            )

        print(
            f"HELD-OUT {MODEL_LABELS[model_name]} | "
            f"AUROC={test_metrics['auc']:.4f} | AUPRC={test_metrics['auprc']:.4f} | "
            f"Sensitivity={test_metrics['sensitivity']:.4f} | Specificity={test_metrics['specificity']:.4f}"
        )

        # ------------------------------------------------------------
        # GRAD-CAM
        # ------------------------------------------------------------
        if GRADCAM_ENABLED:
            final_model = _prepare_model_for_device(final_model, DEVICE, "before_gradcam")
            grad_dir = OUT_DIR / "gradcam" / f"{model_name}_seed{seed}"
            grad_dir.mkdir(parents=True, exist_ok=True)

            ranked = test_slice.copy()
            ranked["predicted"] = (ranked.probability >= FIXED_PATIENT_THRESHOLD).astype(int)
            correct = ranked[ranked.predicted == ranked.label].copy()
            pos = correct[correct.label == 1].sort_values("probability", ascending=False)
            neg = correct[correct.label == 0].sort_values("probability", ascending=True)
            chosen = pd.concat(
                [pos.head(max(1, GRADCAM_MAX_CASES_PER_MODEL // 2)), neg.head(max(1, GRADCAM_MAX_CASES_PER_MODEL // 2))]
            ).drop_duplicates("image_path")

            for case_idx, (_, row) in enumerate(chosen.iterrows(), start=1):
                image_path = row.image_path
                with Image.open(image_path) as im:
                    pil = im.convert("RGB")
                display_image = np.asarray(pil.resize((IMG_SIZE, IMG_SIZE))).astype(np.float32) / 255.0
                tensor = val_transform(pil).float()
                cam, probability = generate_gradcam(final_model, model_name, tensor)
                png_path = grad_dir / f"case_{case_idx:02d}.png"
                pdf_path = grad_dir / f"case_{case_idx:02d}.pdf"
                coverage = save_gradcam_figure(
                    display_image,
                    cam,
                    MODEL_LABELS[model_name],
                    int(row.label),
                    probability,
                    png_path,
                    pdf_path,
                )
                row_out = {
                    "record_type": "gradcam_case",
                    "model": model_name,
                    "model_label": MODEL_LABELS[model_name],
                    "model_group": MODEL_GROUPS[model_name],
                    "seed": seed,
                    "patient_id": row.patient_id,
                    "label": int(row.label),
                    "probability": probability,
                    "image_path": image_path,
                    "saliency_overlay_fraction": coverage,
                    "output_png": str(png_path),
                    "output_pdf": str(pdf_path),
                }
                all_gradcam_case_tables.append(pd.DataFrame([row_out]))
                add_master(**row_out)

            # Quantitative faithfulness on a small, predeclared subset of correctly classified cases.
            if FAITHFULNESS_ENABLED:
                faith_candidates = pd.concat(
                    [pos.head(6), neg.head(6)]
                ).drop_duplicates("image_path").head(FAITHFULNESS_MAX_CASES_PER_MODEL)
                faith_dir = OUT_DIR / "gradcam_faithfulness" / f"{model_name}_seed{seed}"
                faith_dir.mkdir(parents=True, exist_ok=True)
                for case_idx, (_, row) in enumerate(faith_candidates.iterrows(), start=1):
                    with Image.open(row.image_path) as im:
                        pil = im.convert("RGB")
                    tensor = val_transform(pil).float()
                    cam, probability = generate_gradcam(final_model, model_name, tensor)
                    deletion, insertion, deletion_auc, insertion_auc = faithfulness_curves(
                        final_model,
                        tensor.unsqueeze(0),
                        cam,
                        target_class=int(row.label),
                    )
                    curve_df = pd.DataFrame(
                        {
                            "record_type": "gradcam_faithfulness_curve",
                            "model": model_name,
                            "model_label": MODEL_LABELS[model_name],
                            "model_group": MODEL_GROUPS[model_name],
                            "seed": seed,
                            "patient_id": row.patient_id,
                            "label": int(row.label),
                            "image_path": row.image_path,
                            "fraction": FAITHFULNESS_FRACTIONS,
                            "deletion_confidence": deletion,
                            "insertion_confidence": insertion,
                            "deletion_auc": deletion_auc,
                            "insertion_auc": insertion_auc,
                        }
                    )
                    for _, crow in curve_df.iterrows():
                        add_master(**crow.to_dict())
                    fig, ax = plt.subplots(figsize=(7, 6))
                    ax.plot(FAITHFULNESS_FRACTIONS, deletion, marker="o", markersize=3, label=f"Deletion (AUC={deletion_auc:.3f})")
                    ax.plot(FAITHFULNESS_FRACTIONS, insertion, marker="o", markersize=3, label=f"Insertion (AUC={insertion_auc:.3f})")
                    ax.set_xlabel("Fraction of CAM-ranked cells modified/restored")
                    ax.set_ylabel("Target-class confidence")
                    ax.set_title(f"{MODEL_LABELS[model_name]} — Grad-CAM faithfulness")
                    ax.set_xlim(0, 1)
                    ax.set_ylim(0, 1)
                    ax.grid(alpha=0.25)
                    ax.legend()
                    fig.tight_layout()
                    fig.savefig(faith_dir / f"case_{case_idx:02d}.png", dpi=FIG_DPI, bbox_inches="tight", facecolor="white")
                    fig.savefig(faith_dir / f"case_{case_idx:02d}.pdf", bbox_inches="tight", facecolor="white")
                    plt.close(fig)

        del final_model, test_loader
        gc.collect()
        torch.cuda.empty_cache()

# ================================================================
# 18. CONSOLIDATED SUMMARIES
# ================================================================
cv_summary_records = [r for r in master_rows if r.get("record_type") == "cv_fold_summary"]
cv_summary_df = pd.DataFrame(cv_summary_records)

if not cv_summary_df.empty:
    cv_model_summary = (
        cv_summary_df.groupby(["model", "model_label", "model_group", "seed"], as_index=False)
        .agg(
            mean_cv_patient_auc=("validation_patient_auc", "mean"),
            sd_cv_patient_auc=("validation_patient_auc", "std"),
            mean_cv_patient_auprc=("validation_patient_auprc", "mean"),
            sd_cv_patient_auprc=("validation_patient_auprc", "std"),
            mean_best_epoch=("best_epoch", "mean"),
            median_best_epoch=("best_epoch", "median"),
        )
    )
else:
    cv_model_summary = pd.DataFrame()

# Model ranking based only on development CV.
if not cv_model_summary.empty:
    ranking = cv_model_summary.sort_values(
        ["mean_cv_patient_auc", "mean_cv_patient_auprc"], ascending=[False, False]
    ).reset_index(drop=True)
    ranking["cv_rank"] = np.arange(1, len(ranking) + 1)
    selected_model_by_cv = ranking.iloc[0].model
else:
    selected_model_by_cv = None

# Efficiency summary.
efficiency_df = pd.concat(all_efficiency_tables, ignore_index=True) if all_efficiency_tables else pd.DataFrame()

# Test summary extracted from master rows.
test_summary_df = pd.DataFrame(
    [r for r in master_rows if r.get("record_type") == "heldout_test_summary"]
)

# ================================================================
# 19. MASTER CSV — ONE FILE ONLY
# ================================================================
# The master CSV is intentionally long-form. It stores manifests, fold summaries,
# epoch histories, OOF predictions, test predictions, efficiency and Grad-CAM data.
master_csv = OUT_DIR / "MASTER_all_experiment_data.csv"
master_df = pd.DataFrame(master_rows)
master_df.to_csv(master_csv, index=False)

# ================================================================
# 20. REPRODUCIBILITY JSON
# ================================================================
protocol = {
    "protocol_name": "ICH strict patient-level evaluation — device-safe v4",
    "hardware": {
        "device": str(DEVICE),
        "gpu": GPU_NAME,
        "cuda": torch.version.cuda,
        "pytorch": torch.__version__,
    },
    "dataset": {
        "images": int(len(df)),
        "patients": int(df.patient_id.nunique()),
        "hemorrhagic_patients": int(df.loc[df.label == 1, "patient_id"].nunique()),
        "normal_patients": int(df.loc[df.label == 0, "patient_id"].nunique()),
    },
    "outer_split": "9 frozen held-out patients + 36 development patients",
    "outer_split_seed": SEED_SPLIT,
    "test_positive_patients": TEST_POSITIVE_PATIENTS,
    "test_negative_patients": TEST_NEGATIVE_PATIENTS,
    "cross_validation": "4-fold patient-level StratifiedKFold, 27 train / 9 validation patients",
    "models": MODEL_NAMES,
    "model_groups": MODEL_GROUPS,
    "training_seeds": RUN_SEEDS,
    "image_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "normalization_mean": NORMALIZE_MEAN,
    "normalization_std": NORMALIZE_STD,
    "optimizer": OPTIMIZER_NAME,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "dropout": DROPOUT_P,
    "loss": "patient-balanced weighted BCEWithLogitsLoss",
    "scheduler": "ReduceLROnPlateau on validation patient AUROC",
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "primary_endpoint": "patient-level AUROC",
    "secondary_endpoint": "patient-level AUPRC",
    "primary_patient_aggregation": PRIMARY_AGGREGATION,
    "secondary_patient_aggregation": SECONDARY_AGGREGATIONS,
    "fixed_patient_threshold": FIXED_PATIENT_THRESHOLD,
    "high_specificity_target": HIGH_SPEC_TARGET,
    "high_specificity_threshold_source": "development OOF patient predictions only",
    "test_used_for_model_selection": False,
    "test_used_for_epoch_selection": False,
    "test_used_for_threshold_selection": False,
    "bootstrap": "patient-cluster bootstrap",
    "bootstrap_replicates": N_BOOTSTRAP,
    "gradcam_enabled": GRADCAM_ENABLED,
    "evaluation_precision": "FP32",
    "gradcam_precision": "FP32",
    "device_policy": "CUDA required; all training/evaluation/Grad-CAM runs on DEVICE; latency restores original device",
    "gradcam_target_layers": {
        "resnet50": "layer4[-1].conv3",
        "mobilenet_v2": "features[-1][0]",
        "efficientnet_b0": "features[-1][0]",
        "densenet121": "features.denseblock4.denselayer16.conv2",
    },
    "gradcam_overlay": "sparse high-saliency connected regions; not a lesion mask",
    "gradcam_faithfulness": "deletion AUC and insertion AUC",
    "external_validation": False,
    "master_csv": master_csv.name,
}
with open(OUT_DIR / "protocol.json", "w") as f:
    json.dump(protocol, f, indent=2)

# ================================================================
# 21. PUBLICATION FIGURES
# ================================================================
plt.rcParams.update(
    {
        "font.size": 12,
        "axes.titlesize": 15,
        "axes.labelsize": 13,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "legend.fontsize": 10,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
)


def save_fig(fig, stem):
    fig.savefig(OUT_DIR / f"{stem}.png", dpi=FIG_DPI, bbox_inches="tight", facecolor="white")
    fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight", facecolor="white")
    plt.close(fig)

plot_order = MODEL_NAMES
x = np.arange(len(plot_order))

# CV AUROC with non-overlapping labels.
if not cv_model_summary.empty:
    cp = cv_model_summary.set_index("model").reindex(plot_order)
    vals = cp["mean_cv_patient_auc"].to_numpy(float)
    errs = cp["sd_cv_patient_auc"].fillna(0).to_numpy(float)
    fig, ax = plt.subplots(figsize=(9.8, 6.2))
    bars = ax.bar(x, vals, yerr=errs, capsize=6, width=0.62)
    upper = np.minimum(1.0, vals + errs)
    for i, (bar, v, u) in enumerate(zip(bars, vals, upper)):
        label_y = min(1.035, u + 0.035)
        ax.text(i, label_y, f"{v:.3f}", ha="center", va="bottom", fontweight="bold", fontsize=11)
    ax.set_xticks(x)
    ax.set_xticklabels([MODEL_LABELS[m] for m in plot_order])
    ax.set_ylabel("Mean patient-level AUROC")
    ax.set_title("Four-Fold Patient-Level Cross-Validation AUROC")
    ax.set_ylim(0, 1.10)
    ax.grid(axis="y", alpha=0.25)
    save_fig(fig, "FIG1_patient_level_cv_auroc")

# Test AUROC with label placement above CI.
if not test_summary_df.empty:
    tp = test_summary_df.set_index("model").reindex(plot_order)
    vals = tp["auc"].to_numpy(float)
    low = vals - tp["auc_ci_low"].to_numpy(float)
    high = tp["auc_ci_high"].to_numpy(float) - vals
    fig, ax = plt.subplots(figsize=(9.8, 6.2))
    bars = ax.bar(x, vals, yerr=np.vstack([low, high]), capsize=6, width=0.62)
    for i, (bar, v, hi) in enumerate(zip(bars, vals, high)):
        label_y = min(1.035, v + hi + 0.04)
        ax.text(i, label_y, f"{v:.4f}", ha="center", va="bottom", fontweight="bold", fontsize=11)
    ax.set_xticks(x)
    ax.set_xticklabels([MODEL_LABELS[m] for m in plot_order])
    ax.set_ylabel("Held-out patient-level AUROC")
    ax.set_title("Held-Out Patient-Level AUROC (Patient-Cluster Bootstrap 95% CI)")
    ax.set_ylim(0, 1.10)
    ax.grid(axis="y", alpha=0.25)
    save_fig(fig, "FIG2_patient_level_test_auroc")

# Test AUPRC.
if not test_summary_df.empty:
    vals = tp["auprc"].to_numpy(float)
    low = vals - tp["auprc_ci_low"].to_numpy(float)
    high = tp["auprc_ci_high"].to_numpy(float) - vals
    fig, ax = plt.subplots(figsize=(9.8, 6.2))
    bars = ax.bar(x, vals, yerr=np.vstack([low, high]), capsize=6, width=0.62)
    for i, (bar, v, hi) in enumerate(zip(bars, vals, high)):
        label_y = min(1.035, v + hi + 0.04)
        ax.text(i, label_y, f"{v:.4f}", ha="center", va="bottom", fontweight="bold", fontsize=11)
    ax.set_xticks(x)
    ax.set_xticklabels([MODEL_LABELS[m] for m in plot_order])
    ax.set_ylabel("Held-out patient-level AUPRC")
    ax.set_title("Held-Out Patient-Level AUPRC (Patient-Cluster Bootstrap 95% CI)")
    ax.set_ylim(0, 1.10)
    ax.grid(axis="y", alpha=0.25)
    save_fig(fig, "FIG3_patient_level_test_auprc")

# OOF ROC curves.
oof_all = pd.concat(all_oof_patient_tables, ignore_index=True) if all_oof_patient_tables else pd.DataFrame()
if not oof_all.empty:
    fig, ax = plt.subplots(figsize=(8, 7))
    for model_name in plot_order:
        g = oof_all[oof_all.model == model_name]
        fpr, tpr, _ = roc_curve(g.label, g.mean_probability)
        auc = roc_auc_score(g.label, g.mean_probability)
        ax.plot(fpr, tpr, linewidth=2.2, label=f"{MODEL_LABELS[model_name]} (AUROC={auc:.3f})")
    ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1.2)
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title("Development OOF Patient-Level ROC Curves")
    ax.legend(loc="lower right")
    ax.grid(alpha=0.25)
    save_fig(fig, "FIG4_patient_level_oof_roc")

# Test ROC curves.
test_all = pd.concat(all_test_patient_tables, ignore_index=True) if all_test_patient_tables else pd.DataFrame()
if not test_all.empty:
    fig, ax = plt.subplots(figsize=(8, 7))
    for model_name in plot_order:
        g = test_all[test_all.model == model_name]
        fpr, tpr, _ = roc_curve(g.label, g.mean_probability)
        auc = roc_auc_score(g.label, g.mean_probability)
        ax.plot(fpr, tpr, linewidth=2.2, label=f"{MODEL_LABELS[model_name]} (AUROC={auc:.3f})")
    ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1.2)
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title("Held-Out Patient-Level ROC Curves")
    ax.legend(loc="lower right")
    ax.grid(alpha=0.25)
    save_fig(fig, "FIG5_patient_level_test_roc")

# Efficiency: parameters.
if not efficiency_df.empty:
    ep = efficiency_df.groupby(["model", "model_label"], as_index=False)["parameters_million"].mean().set_index("model").reindex(plot_order)
    fig, ax = plt.subplots(figsize=(9.8, 6.2))
    vals = ep.parameters_million.to_numpy(float)
    bars = ax.bar(x, vals, width=0.62)
    for i, (bar, v) in enumerate(zip(bars, vals)):
        ax.text(i, v + max(vals) * 0.02, f"{v:.2f}M", ha="center", va="bottom", fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels([MODEL_LABELS[m] for m in plot_order])
    ax.set_ylabel("Parameters (millions)")
    ax.set_title("Model Parameter Count")
    ax.grid(axis="y", alpha=0.25)
    save_fig(fig, "FIG6_model_parameters")

# Efficiency: CPU latency.
# FIX: read the flattened scalar column "cpu_latency_mean_ms" instead of the
# old dict-valued "cpu_latency_ms_per_image" column (see Section 17 note).
if not efficiency_df.empty and efficiency_df["cpu_latency_mean_ms"].notna().any():
    ep = efficiency_df.groupby("model")["cpu_latency_mean_ms"].mean().reindex(plot_order)
    fig, ax = plt.subplots(figsize=(9.8, 6.2))
    vals = ep.to_numpy(float)
    bars = ax.bar(x, vals, width=0.62)
    finite = vals[np.isfinite(vals)]
    vmax = finite.max() if len(finite) else 1
    for i, (bar, v) in enumerate(zip(bars, vals)):
        if np.isfinite(v):
            ax.text(i, v + vmax * 0.02, f"{v:.2f}", ha="center", va="bottom", fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels([MODEL_LABELS[m] for m in plot_order])
    ax.set_ylabel("CPU inference latency (ms/image)")
    ax.set_title("CPU Inference Latency")
    ax.grid(axis="y", alpha=0.25)
    save_fig(fig, "FIG7_cpu_latency")

# ================================================================
# 22. SAVE SUMMARY TABLES AS ADDITIONAL NON-CSV ARTIFACTS
# ================================================================
# HTML keeps the ZIP self-contained while the requested tabular source remains ONE CSV.
if not cv_model_summary.empty:
    cv_model_summary.to_html(OUT_DIR / "TABLE1_cv_summary.html", index=False, float_format=lambda x: f"{x:.4f}")
if not test_summary_df.empty:
    test_summary_df.to_html(OUT_DIR / "TABLE2_test_summary.html", index=False, float_format=lambda x: f"{x:.4f}")
if not efficiency_df.empty:
    efficiency_df.to_html(OUT_DIR / "TABLE3_efficiency.html", index=False, float_format=lambda x: f"{x:.4f}")

# ================================================================
# 23. FINAL CONSOLE SUMMARY
# ================================================================
print("\n" + "=" * 120)
print("FINAL DEVELOPMENT CV SUMMARY")
print("=" * 120)
if not cv_model_summary.empty:
    print(cv_model_summary.round(4).to_string(index=False))
print(f"Selected model by development patient AUROC: {selected_model_by_cv}")

print("\n" + "=" * 120)
print("FINAL HELD-OUT TEST SUMMARY")
print("=" * 120)
if not test_summary_df.empty:
    print(
        test_summary_df[
            ["model_label", "test_patients", "test_slices", "auc", "auc_ci_low", "auc_ci_high", "auprc", "auprc_ci_low", "auprc_ci_high", "accuracy", "sensitivity", "specificity", "f1"]
        ].round(4).to_string(index=False)
    )

print("\n" + "=" * 120)
print("MASTER CSV")
print("=" * 120)
print(master_csv)
print(f"Rows: {len(master_df):,} | Columns: {len(master_df.columns)}")

# ================================================================
# 24. ZIP EVERYTHING — ONE DOWNLOAD
# ================================================================
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as zf:
    # Generated outputs.
    for path in OUT_DIR.rglob("*"):
        if path.is_file():
            zf.write(
                path,
                arcname=Path("experiment_output") / path.relative_to(OUT_DIR),
            )

    # Optional original source CT images. Stream them directly from /kaggle/input.
    source_count = 0
    if INCLUDE_SOURCE_IMAGES_IN_ZIP:
        for source_root, archive_root in [
            (
                Path(HEMORRHAGIC_DIR),
                Path("source_data") / "Hemorrhagic" / "KANAMA",
            ),
            (
                Path(NORMAL_DIR),
                Path("source_data") / "NORMAL",
            ),
        ]:
            if not source_root.exists():
                raise FileNotFoundError(
                    f"Source dataset directory not found while creating ZIP: {source_root}"
                )
            for source_file in source_root.rglob("*"):
                if source_file.is_file() and source_file.suffix.lower() in VALID_EXTENSIONS:
                    zf.write(
                        source_file,
                        arcname=archive_root / source_file.relative_to(source_root),
                    )
                    source_count += 1

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    bad = zf.testzip()
    if bad is not None:
        raise RuntimeError(f"ZIP integrity check failed at {bad}")
    zip_member_count = len(zf.namelist())

print("\n" + "=" * 120)
print("COMPLETE ZIP CREATED")
print("=" * 120)
print(f"ZIP: {ZIP_PATH}")
print(f"Size: {ZIP_PATH.stat().st_size / (1024**2):.2f} MB")
print(f"Files inside ZIP: {zip_member_count}")
print(f"Original source images added: {source_count if INCLUDE_SOURCE_IMAGES_IN_ZIP else 0}")
print("Contains: one master CSV, JSON protocol, model checkpoints, PNG/PDF figures, Grad-CAM and faithfulness outputs, plus source CT images when enabled.")
print("The notebook must be configured to a GPU runtime; this script uses CUDA when available and refuses CPU execution.")
print("=" * 120)

ICH PATIENT-LEVEL DEEP LEARNING — FINAL PUBLICATION PIPELINE — DEVICE-SAFE
CUDA available : True
GPU            : Tesla T4
CUDA version   : 12.8
PyTorch        : 2.10.0+cu128
Models         : ['MobileNetV2', 'EfficientNet-B0', 'ResNet-50', 'DenseNet-121']
Seeds          : [42]

DATASET AUDIT
Total slices          : 6795
Hemorrhagic slices    : 2690
Normal slices         : 4105
Total patients        : 45
Hemorrhagic patients  : 18
Normal patients       : 27
count     45.000000
mean     151.000000
std       61.746549
min       23.000000
25%      127.000000
50%      132.000000
75%      196.000000
max      360.000000
dtype: float64

OUTER PATIENT SPLIT
Development : 36 patients
Held-out    : 9 patients
Held-out IDs: ['HEMORRHAGIC_1[1]', 'HEMORRHAGIC_2[2]', 'HEMORRHAGIC_6[6]', 'HEMORRHAGIC_9[9]', 'NORMAL_N5[N5]', 'NORMAL_N9[N9]', 'NORMAL_N14[N14]', 'NORMAL_N22[N22]', 'NORMAL_N23[N23]']


FAIL-FAST PREFLIGHT
Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /roo

100%|██████████| 13.6M/13.6M [00:00<00:00, 110MB/s] 


Preflight: CPU-latency device restoration PASS | CPU smoke latency=273.79 ms | restored=('cuda', 0)
Preflight: MobileNetV2
  PASS | forward=(1, 1) | Grad-CAM=(224, 224) | P(ICH)=0.4517
Preflight: EfficientNet-B0
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 128MB/s] 


  PASS | forward=(1, 1) | Grad-CAM=(224, 224) | P(ICH)=0.4734
Preflight: ResNet-50
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 217MB/s] 


  PASS | forward=(1, 1) | Grad-CAM=(224, 224) | P(ICH)=0.4472
Preflight: DenseNet-121
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 165MB/s] 


  PASS | forward=(1, 1) | Grad-CAM=(224, 224) | P(ICH)=0.4665
PREFLIGHT PASSED: device, four model forwards, Grad-CAM targets, and split checks.
No expensive model training begins before these checks pass.

MODEL: MobileNetV2 | SEED: 42
MobileNetV2      | fold 1 | epoch 001 | val patient AUROC=0.7000 | AUPRC=0.7986
MobileNetV2      | fold 1 | epoch 002 | val patient AUROC=0.6500 | AUPRC=0.7500
MobileNetV2      | fold 1 | epoch 003 | val patient AUROC=0.7500 | AUPRC=0.7929
MobileNetV2      | fold 1 | epoch 004 | val patient AUROC=0.8000 | AUPRC=0.8304
MobileNetV2      | fold 1 | epoch 005 | val patient AUROC=0.8000 | AUPRC=0.8167
MobileNetV2      | fold 1 | epoch 006 | val patient AUROC=0.8000 | AUPRC=0.8304
MobileNetV2      | fold 1 | epoch 007 | val patient AUROC=0.8500 | AUPRC=0.8542
MobileNetV2      | fold 1 | epoch 008 | val patient AUROC=0.6000 | AUPRC=0.7321
MobileNetV2      | fold 1 | epoch 009 | val patient AUROC=0.8500 | AUPRC=0.8929
MobileNetV2      | fold 1 | epoch 010 | val